In [1]:
import time

import torch
import triton
import triton.language as tl

In [2]:
def is_cuda():
    return triton.runtime.driver.active.get_current_target().backend == "cuda"

In [3]:
def is_hip_mi200():
    target = triton.runtime.driver.active.get_current_target()
    return target.backend == 'hip' and target.arch == 'gfx90a'

In [4]:
is_cuda()

True

In [5]:
"""
PA2 Part 2: MatMul+Relu+Add Fused Optimization.
The kernel uses several optimization techniques:

  1. Shared memory tiling.
  2. Register tiling.
  3. Cooperative fetching.
  4. Operator Fusion
  5. Write cache / epilogue fusion.

Fill in the missing parts (marked with TODO).
"""

# -----------------------------------------------------------------------------
# Tiling parameters - autotuned below. Tile sizes kept small so that the grid
# fills enough SMs on the small benchmark shape (128x64 @ 64x256) on T4.
# -----------------------------------------------------------------------------
_AUTOTUNE_CONFIGS = [
    triton.Config({'BLOCK_M': 16, 'BLOCK_N': 32, 'BLOCK_K': 32}, num_warps=2, num_stages=2),
    triton.Config({'BLOCK_M': 16, 'BLOCK_N': 64, 'BLOCK_K': 32}, num_warps=2, num_stages=2),
    triton.Config({'BLOCK_M': 32, 'BLOCK_N': 32, 'BLOCK_K': 32}, num_warps=2, num_stages=2),
    triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64, 'BLOCK_K': 32}, num_warps=2, num_stages=2),
    triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64, 'BLOCK_K': 32}, num_warps=4, num_stages=2),
    triton.Config({'BLOCK_M': 32, 'BLOCK_N': 64, 'BLOCK_K': 64}, num_warps=4, num_stages=2),
    triton.Config({'BLOCK_M': 32, 'BLOCK_N': 128, 'BLOCK_K': 32}, num_warps=4, num_stages=2),
    triton.Config({'BLOCK_M': 64, 'BLOCK_N': 32, 'BLOCK_K': 32}, num_warps=2, num_stages=2),
    triton.Config({'BLOCK_M': 64, 'BLOCK_N': 64, 'BLOCK_K': 32}, num_warps=4, num_stages=2),
    triton.Config({'BLOCK_M': 64, 'BLOCK_N': 64, 'BLOCK_K': 64}, num_warps=4, num_stages=2),
    triton.Config({'BLOCK_M': 64, 'BLOCK_N': 128, 'BLOCK_K': 32}, num_warps=4, num_stages=2),
    triton.Config({'BLOCK_M': 128, 'BLOCK_N': 64, 'BLOCK_K': 32}, num_warps=4, num_stages=2),
    triton.Config({'BLOCK_M': 128, 'BLOCK_N': 128, 'BLOCK_K': 32}, num_warps=8, num_stages=2),
]


# -----------------------------------------------------------------------------
# Triton Kernel: Matrix Multiplication + ReLU + Add
#
# The kernel uses:
#   Step 1: Tile assignment (each kernel computes a tile of C)
#   Step 2: Shared memory tiling + Cooperative Fetching: Load tiles of A and B.
#   Step 3: Register tiling: Use a register accumulator.
#   Step 4: Add and ReLU fusion
#   Step 5: Write cache/Epilogue: Write the final tile back to global memory.
# -----------------------------------------------------------------------------
@triton.jit
def matmul_add_relu_kernel_fp16(
        a_ptr, b_ptr, c_ptr, d_ptr,
        M, N, K,
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        stride_dm, stride_dn,
        BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    # -------------------------------------------------------------------------
    # Step 1: Tile: Assignment
    #
    # Each kernel instance is mapped to a tile in the output matrix C.
    # Compute the starting indices (m_start, n_start) for this tile.
    # -------------------------------------------------------------------------
    # TODO: Compute the tile indices using program_id(0) for M and program_id(1) for N.
    # D = ReLU(A @ B + C)
    # A: [M x K]
    # B: [K x N]
    # C: [M x N]
    # D: [M x N]

    # every thread process one block of D
    pid_m = tl.program_id(axis=0)
    pid_n = tl.program_id(axis=1)

    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)

    # -------------------------------------------------------------------------
    # Step 2: Register Tiling
    # -------------------------------------------------------------------------
    # TODO: Initialize the accumulator "acc" with zeros (dtype: float16).
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float16)

    # -------------------------------------------------------------------------
    # Step 3: Shared Memory Tiling & Cooperative Fetching.
    # Compute pointers to the sub-tiles of A and B that are needed to compute
    # the current C tile. The offsets here serve to load BLOCK_SIZE_M x BLOCK_SIZE_K
    # and BLOCK_SIZE_K x BLOCK_SIZE_N blocks from A and B respectively.
    # -------------------------------------------------------------------------
    # TODO: Finish code below
    a_ptrs = a_ptr + (offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)

    mask_m = offs_m[:, None] < M
    mask_n = offs_n[None, :] < N

    for k in range(0, K, BLOCK_K):
        mask_ak = (offs_k[None, :] + k) < K
        mask_bk = (offs_k[:, None] + k) < K
        a = tl.load(a_ptrs, mask=mask_m & mask_ak, other=0.0)
        b = tl.load(b_ptrs, mask=mask_bk & mask_n, other=0.0)
        acc = tl.dot(a, b, acc=acc, out_dtype=tl.float16)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk

    # -------------------------------------------------------------------------
    # Step 4: Apply ReLU and Add C to the accumulator
    # -------------------------------------------------------------------------
    # TODO: Finish code below
    c_ptrs = c_ptr + (offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn)
    mask_mn = mask_m & mask_n
    c = tl.load(c_ptrs, mask=mask_mn, other=0.0)
    z = acc + c
    d = tl.where(z > 0, z, 0.0)

    # -------------------------------------------------------------------------
    # Step 5: Write Cache / Epilogue Fusion: Write the computed tile to D.
    # -------------------------------------------------------------------------
    # TODO: Finish code below
    d_ptrs = d_ptr + (offs_m[:, None] * stride_dm + offs_n[None, :] * stride_dn)
    tl.store(d_ptrs, d, mask=mask_mn)


# Hardcoded tile config (winner from the grid search cell). Tweak these.
_BLOCK_M = 32
_BLOCK_N = 16
_BLOCK_K = 32
_NUM_WARPS = 2
_NUM_STAGES = 3


def matmul_add_relu_fp16(a: torch.Tensor, b: torch.Tensor, c: torch.Tensor) -> torch.Tensor:
    """
    Computes Output = ReLU(A @ B + C) using fp16 precision for maximum throughput.
    """
    M, K = a.shape
    N = b.shape[1]
    d = torch.empty((M, N), device=a.device, dtype=torch.float16)
    grid = (triton.cdiv(M, _BLOCK_M), triton.cdiv(N, _BLOCK_N))
    matmul_add_relu_kernel_fp16[grid](
        a, b, c, d,
        M, N, K,
        a.stride(0), a.stride(1),
        b.stride(0), b.stride(1),
        c.stride(0), c.stride(1),
        d.stride(0), d.stride(1),
        BLOCK_M=_BLOCK_M, BLOCK_N=_BLOCK_N, BLOCK_K=_BLOCK_K,
        num_warps=_NUM_WARPS, num_stages=_NUM_STAGES,
    )
    return d

In [6]:
# Reference implementation using PyTorch
def reference_matmul_add_relu(A, B, C):
    result = torch.matmul(A, B).add(C).relu_()
    return result

In [7]:
# -----------------------------------------------------------------------------
# Accuracy Tests
# -----------------------------------------------------------------------------
if __name__ == "__main__":
    torch.manual_seed(0)
    a = torch.randn((512, 512), device=torch.device("cuda"), dtype=torch.float16)
    b = torch.randn((512, 512), device=torch.device("cuda"), dtype=torch.float16)
    c = torch.randn((512, 512), device=torch.device("cuda"), dtype=torch.float16)
    triton_output = matmul_add_relu_fp16(a, b, c)
    torch_output = reference_matmul_add_relu(a, b, c)
    print(f"triton_output_with_fp16_inputs={triton_output}")
    print(f"torch_output_with_fp16_inputs={torch_output}")
    rtol = 1e-2 if is_hip_mi200() else 0.032
    if torch.allclose(triton_output, torch_output, atol=0.15, rtol=rtol):
        print("✅ Triton and Torch match")
    else:
        diff = triton_output - torch_output
        abs_diff = torch.abs(diff)
        max_abs_diff = torch.max(abs_diff)
        print(f"❌ Triton and Torch differ: {max_abs_diff=}")

triton_output_with_fp16_inputs=tensor([[ 0.0000,  0.0000, 16.5469,  ...,  0.0000, 16.0000,  0.0000],
        [24.0781,  1.7754,  0.0000,  ...,  7.7109,  4.0234, 33.3750],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [ 0.0000, 41.9062, 32.4375,  ..., 37.8125,  0.0000,  0.0000],
        [42.0625,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [29.5469,  0.0000,  0.0000,  ...,  0.9336,  2.2246, 12.2656]],
       device='cuda:0', dtype=torch.float16)
torch_output_with_fp16_inputs=tensor([[ 0.0000,  0.0000, 16.5469,  ...,  0.0000, 16.0156,  0.0000],
        [24.0938,  1.7695,  0.0000,  ...,  7.7109,  4.0391, 33.4375],
        [ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        ...,
        [ 0.0000, 41.9062, 32.4062,  ..., 37.8438,  0.0000,  0.0000],
        [42.0938,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
        [29.5469,  0.0000,  0.0000,  ...,  0.9277,  2.2148, 12.2578]],
       device='cuda:0', dt

In [12]:
# -----------------------------------------------------------------------------
# Performance Benchmark 
# IMPORTANT: DO NOT CHANGE THIS CODE. 
# THIS IS THE EXACT CODE THAT WILL BE USED TO GRADE YOUR IMPLEMENTATION.
# ANY CHANGES TO THIS CODE (INCLUDING DIMENSIONS, REPEATS, etc.)
# WILL CAUSE YOU TO HAVE DIFFERENT SPEEDUP RESULTS.
# -----------------------------------------------------------------------------
M = 512
K = 512
N = 512

# Torch reference: 12.42 us
# Top 10 configs (BLOCK_M, BLOCK_N, BLOCK_K, num_warps, num_stages):
#      8.23 us  speedup= 1.51x  BM= 16 BN=128 BK= 16 warps=4 stages=2
#      8.24 us  speedup= 1.51x  BM= 16 BN=128 BK= 32 warps=8 stages=2
#      8.24 us  speedup= 1.51x  BM= 16 BN=128 BK= 32 warps=2 stages=2
#      8.25 us  speedup= 1.51x  BM= 16 BN=128 BK= 16 warps=8 stages=3
#      8.25 us  speedup= 1.51x  BM= 16 BN=128 BK= 16 warps=2 stages=2
#      8.25 us  speedup= 1.51x  BM= 16 BN= 32 BK= 64 warps=4 stages=2
#      8.25 us  speedup= 1.51x  BM= 16 BN=128 BK= 64 warps=2 stages=2
#      8.26 us  speedup= 1.51x  BM=128 BN=128 BK= 64 warps=8 stages=3
#      8.26 us  speedup= 1.50x  BM= 16 BN= 64 BK= 64 warps=2 stages=2
#      8.26 us  speedup= 1.50x  BM= 16 BN= 64 BK= 64 warps=2 stages=3

# KEEP THESE MATRICES IN FP16. FP32 WILL NOT PROVIDE ACCURATE RESULTS
A = torch.randn((M, K), device="cuda", dtype=torch.float16)
B = torch.randn((K, N), device="cuda", dtype=torch.float16)
C = torch.randn((M, N), device="cuda", dtype=torch.float16)

# warmup
_ = matmul_add_relu_fp16(A, B, C)
_ = reference_matmul_add_relu(A, B, C)

REPEATS = 5000

# time your implementation
print("Triton implementation")
torch.cuda.synchronize()
start = time.perf_counter()
for _ in range(REPEATS):
    _ = matmul_add_relu_fp16(A, B, C)
torch.cuda.synchronize()
triton_time = (time.perf_counter() - start) / REPEATS

# time pytorch
print("PyTorch implementation")
torch.cuda.synchronize()
start = time.perf_counter()
for _ in range(REPEATS):
    _ = reference_matmul_add_relu(A, B, C)
torch.cuda.synchronize()
torch_time = (time.perf_counter() - start) / REPEATS

print(f"Performance comparison for matrix multiplication ({M}x{K} @ {K}x{N}):")
print(f"Triton implementation: {triton_time * 1000:.4f} ms")
print(f"PyTorch implementation: {torch_time * 1000:.4f} ms")

print(f"\nSpeedup of Triton vs PyTorch: {torch_time / triton_time:.2f}x")

Triton implementation
PyTorch implementation
Performance comparison for matrix multiplication (512x512 @ 512x512):
Triton implementation: 0.0125 ms
PyTorch implementation: 0.0124 ms

Speedup of Triton vs PyTorch: 0.99x


In [10]:
# -----------------------------------------------------------------------------
# Grid search over (BLOCK_M, BLOCK_N, BLOCK_K, num_warps, num_stages).
# Each config is launched directly (bypassing autotune) and timed on the
# benchmark shape M=128, K=64, N=256.
# -----------------------------------------------------------------------------
import itertools

GS_M, GS_K, GS_N = 32, 32, 16  # match the benchmark shape (M, K, N)
gs_A = torch.randn((GS_M, GS_K), device="cuda", dtype=torch.float16)
gs_B = torch.randn((GS_K, GS_N), device="cuda", dtype=torch.float16)
gs_C = torch.randn((GS_M, GS_N), device="cuda", dtype=torch.float16)


@triton.jit
def _gs_kernel(
        a_ptr, b_ptr, c_ptr, d_ptr,
        M, N, K,
        stride_am, stride_ak,
        stride_bk, stride_bn,
        stride_cm, stride_cn,
        stride_dm, stride_dn,
        BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    pid_m = tl.program_id(0)
    pid_n = tl.program_id(1)
    offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)
    offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)
    offs_k = tl.arange(0, BLOCK_K)
    a_ptrs = a_ptr + (offs_m[:, None] * stride_am + offs_k[None, :] * stride_ak)
    b_ptrs = b_ptr + (offs_k[:, None] * stride_bk + offs_n[None, :] * stride_bn)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float16)
    mask_m = offs_m[:, None] < M
    mask_n = offs_n[None, :] < N
    for k in range(0, K, BLOCK_K):
        mask_ak = (offs_k[None, :] + k) < K
        mask_bk = (offs_k[:, None] + k) < K
        a = tl.load(a_ptrs, mask=mask_m & mask_ak, other=0.0)
        b = tl.load(b_ptrs, mask=mask_bk & mask_n, other=0.0)
        acc = tl.dot(a, b, acc=acc, out_dtype=tl.float16)
        a_ptrs += BLOCK_K * stride_ak
        b_ptrs += BLOCK_K * stride_bk
    c_ptrs = c_ptr + (offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn)
    mask_mn = mask_m & mask_n
    c = tl.load(c_ptrs, mask=mask_mn, other=0.0)
    z = acc + c
    d = tl.where(z > 0, z, 0.0)
    d_ptrs = d_ptr + (offs_m[:, None] * stride_dm + offs_n[None, :] * stride_dn)
    tl.store(d_ptrs, d, mask=mask_mn)


def _run_config(bm, bn, bk, nw, ns, repeats=2000):
    d = torch.empty((GS_M, GS_N), device="cuda", dtype=torch.float16)
    grid = (triton.cdiv(GS_M, bm), triton.cdiv(GS_N, bn))
    args = (
        gs_A, gs_B, gs_C, d,
        GS_M, GS_N, GS_K,
        gs_A.stride(0), gs_A.stride(1),
        gs_B.stride(0), gs_B.stride(1),
        gs_C.stride(0), gs_C.stride(1),
        d.stride(0), d.stride(1),
    )
    # warmup + compile
    for _ in range(5):
        _gs_kernel[grid](*args, BLOCK_M=bm, BLOCK_N=bn, BLOCK_K=bk, num_warps=nw, num_stages=ns)
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(repeats):
        _gs_kernel[grid](*args, BLOCK_M=bm, BLOCK_N=bn, BLOCK_K=bk, num_warps=nw, num_stages=ns)
    torch.cuda.synchronize()
    return (time.perf_counter() - start) / repeats


# Reference torch time on the same shape, for context.
for _ in range(5):
    _ = reference_matmul_add_relu(gs_A, gs_B, gs_C)
torch.cuda.synchronize()
_t0 = time.perf_counter()
for _ in range(2000):
    _ = reference_matmul_add_relu(gs_A, gs_B, gs_C)
torch.cuda.synchronize()
torch_ref_time = (time.perf_counter() - _t0) / 2000

block_ms = [16, 32, 64, 128]
block_ns = [16, 32, 64, 128]
block_ks = [16, 32, 64]
num_warps_list = [2, 4, 8]
num_stages_list = [2, 3]

results = []
for bm, bn, bk, nw, ns in itertools.product(
        block_ms, block_ns, block_ks, num_warps_list, num_stages_list
):
    # Skip configs that are clearly too large / unsupported on T4.
    if bm * bn > 128 * 128:
        continue
    try:
        t = _run_config(bm, bn, bk, nw, ns, repeats=1000)
        results.append((t, bm, bn, bk, nw, ns))
    except Exception as e:
        # Some configs may fail to compile (e.g. shared memory limits).
        pass

results.sort()
print(f"Torch reference: {torch_ref_time * 1e6:.2f} us")
print("Top 10 configs (BLOCK_M, BLOCK_N, BLOCK_K, num_warps, num_stages):")
for t, bm, bn, bk, nw, ns in results[:10]:
    print(f"  {t * 1e6:7.2f} us  speedup={torch_ref_time / t:5.2f}x  "
          f"BM={bm:3d} BN={bn:3d} BK={bk:3d} warps={nw} stages={ns}")

Torch reference: 12.42 us
Top 10 configs (BLOCK_M, BLOCK_N, BLOCK_K, num_warps, num_stages):
     8.23 us  speedup= 1.51x  BM= 16 BN=128 BK= 16 warps=4 stages=2
     8.24 us  speedup= 1.51x  BM= 16 BN=128 BK= 32 warps=8 stages=2
     8.24 us  speedup= 1.51x  BM= 16 BN=128 BK= 32 warps=2 stages=2
     8.25 us  speedup= 1.51x  BM= 16 BN=128 BK= 16 warps=8 stages=3
     8.25 us  speedup= 1.51x  BM= 16 BN=128 BK= 16 warps=2 stages=2
     8.25 us  speedup= 1.51x  BM= 16 BN= 32 BK= 64 warps=4 stages=2
     8.25 us  speedup= 1.51x  BM= 16 BN=128 BK= 64 warps=2 stages=2
     8.26 us  speedup= 1.51x  BM=128 BN=128 BK= 64 warps=8 stages=3
     8.26 us  speedup= 1.50x  BM= 16 BN= 64 BK= 64 warps=2 stages=2
     8.26 us  speedup= 1.50x  BM= 16 BN= 64 BK= 64 warps=2 stages=3
